# Intro to LLMs and PydanticAI

## 1. Koppla upp en modell

`load_dotenv()` läser `.env` och lägger nycklarna i miljövariablerna. `os.environ["NVIDIA_API_KEY"]` hämtar nyckeln och ger `KeyError` om den saknas, så kolla `.env` först om cellen kraschar.

`OpenAIChatModel` betyder inte att vi använder OpenAI. Det är formatet för anropen. `OpenAIProvider` med `base_url` pekar på NVIDIA:s server, som förstår samma format.

Att tänka på:
- `.env` får aldrig hamna i git.
- `model_name` måste vara ett namn som just den leverantören har.
- Byter vi till OpenRouter ändras bara `base_url`, nyckeln och `model_name`. Resten av koden är densamma.
- Sista raden `model` skriver bara ut objektet, så att vi ser vilken modell som är vald.

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
import os
from dotenv import load_dotenv

load_dotenv()

model = OpenAIChatModel(
    model_name ="nvidia/nemotron-3.5-lightning-30b-a3b",
    provider = OpenAIProvider(
        base_url = "https://integrate.api.nvidia.com/v1", api_key =os.environ["NVIDIA_API_KEY"]
    ),
)
model

## 2. Skapa en agent och ställ en fråga

`system_prompt` styr hur agenten beter sig i alla svar (ton, längd, stil). `prompt` är själva frågan.

`agent.run()` är asynkron, därför `await`. Det fungerar direkt i en notebook. I en vanlig `.py`-fil används `agent.run_sync(prompt)` i stället.

`result.output` är svaret. Utan `output_type` är det alltid en sträng, alltså fritext. Med `output_type` (structured output) får vi i stället ett pydantic-objekt med fasta fält, och det är det vi behöver till quizet.

Att tänka på: ändra `system_prompt` och kör igen, så ser du hur mycket den styr svaret.

In [ ]:
agent = Agent(model=model, system_prompt="You are funny joking bot, answer shortly in a nerdy style, also add emojis")

prompt = "Explain what PydanticAI is and the difference between it and Pydantic"

result = await agent.run(prompt)
print(result.output)

## 3. Alla meddelanden i samtalet

`all_messages()` visar allt som skickades fram och tillbaka mellan oss och modellen, inte bara svaret. Det är så man ser vad agenten faktiskt gjorde.

In [ ]:
result.all_messages()

## 4. Hur många meddelanden?

Det är en lista. Vid en enkel fråga är den oftast 2 lång:
1. `ModelRequest`: det vi skickade (system prompt + fråga)
2. `ModelResponse`: modellens svar

När agenten använder verktyg blir listan längre: ett anrop till verktyget, verktygets svar, och sedan modellens slutliga svar.

In [ ]:
type(result.all_messages()), len(result.all_messages())

## 5. Första meddelandet: vår förfrågan

Index 0 är `ModelRequest`, det vi skickade till modellen.

In [ ]:
result.all_messages()[0]

## 6. Delarna i förfrågan

Varje meddelande består av delar (`parts`), t.ex. `SystemPromptPart` (system prompten) och `UserPromptPart` (frågan).

Visa typen på varje del i stället för att gissa ordningen:

    [p.part_kind for p in result.all_messages()[0].parts]

In [ ]:
result.all_messages()[0].parts, len(result.all_messages()[0].parts)

## 7. Varför `parts[1].content` ger fel

I vår version av PydanticAI är `parts[1]` en `ToolAvailabilityDeltaPart`, inte frågan. Den delen har ingen `.content`, därför felet. Enligt namnet beskriver den vilka verktyg agenten har tillgång till.

Lärdom: ordningen på delarna kan ändras mellan versioner av biblioteket. Hämta delen på typ, inte på position:

    [p.content for p in result.all_messages()[0].parts if p.part_kind == "user-prompt"]

In [ ]:
result.all_messages()[0].parts[0].content

## 8. Delarna i svaret

Index 1 är `ModelResponse`, modellens svar. `.__len__()` gör samma sak som `len(...)`. Använd `len()`, det är det vanliga sättet.

Svaret kan ha fler än en del. En modell som "tänker" innan den svarar kan ge en `ThinkingPart` före `TextPart` med själva svaret.

In [ ]:
result.all_messages()[1].parts.__len__()

## 9. En del i svaret

Samma sak som i cell 7: kontrollera typen innan du läser innehållet, eftersom positionen beror på modell och version.

    [p.part_kind for p in result.all_messages()[1].parts]

Bonus: `result.usage()` visar hur många tokens frågan kostade.

In [ ]:
result.all_messages()[1].parts[1]

## Tokens

## Tokens: vad frågan kostade

`result.usage` visar hur många tokens körningen använde. Tokens är det leverantören räknar och tar betalt för.

| Fält | Värde | Betyder |
| --- | --- | --- |
| `input_tokens` | 53 | det vi skickade: system prompt + fråga |
| `output_tokens` | 951 | allt modellen skrev, inklusive tänkandet |
| `output_reasoning_tokens` | 821 | det modellen "tänkte" innan den svarade, syns inte i svaret |
| `requests` | 1 | antal anrop till modellen |

Bara ca 130 av 951 output-tokens (951 − 821) blev själva svaret. Resten var resonemang. En resonerande modell som Nemotron är därför långsammare och dyrare per svar än svaret ser ut att vara.

Att tänka på:
- Fler meddelanden, verktygsanrop och längre system prompt ökar `input_tokens`.
- Använder agenten verktyg blir `requests` större än 1, ett anrop per runda.
- Till quizet: mät tokens och tid per fråga. Det blir ett konkret mått på kostnad när vi jämför modeller eller körningar med och utan RAG.

In [ ]:
result.usage

# ChefBot

## ChefBot – agent med instruktioner och svarsformat

Här skapas en `Agent` med en **persona** och ett **fast svarsformat** via `instructions`:
- **Persona:** en kock på en pizzeria i en svensk förort, med fokus på italiensk mat.
- **Språk:** "swenglish", alltså en blandning av svenska och engelska.
- **Format:** beskrivning (1–2 meningar), ingredienser i punktlista och kort steg-för-steg.

`instructions` motsvarar **system-rollen** i chattformatet. Den styr hur modellen beter sig vid varje anrop. `agent.run()` skickar användarens fråga som **user-rollen**.

`await` behövs eftersom `run()` är asynkron. I Jupyter går det att använda `await` direkt i en cell.

In [ ]:
chef_agent = Agent(
    model=model,
    instructions=("You are a professional chef at a pizza place in a suburb in Sweden that are very good at making italian food and explaining in a pedagogical way how to create yummy dishes"
    """
                    Answer in the following format, with swenglish language:

                    Description 1-2 sentences of what the dish is
        
                    List of ingredients in bullet points

                    Short step by step description on how to make the dish
    """
    ),
)

result = await chef_agent.run("Give me 3 dishes in a list")
print(result.output)

### Spara svaret som markdown-fil

`result.output` innehåller modellens svar som en sträng, och den skrivs till `dishes.md`.

`encoding="utf-8"` krävs på Windows. Där är standardkodningen `cp1252`, som inte klarar alla tecken som LLM:er ofta genererar, t.ex. `→` och emojis. Utan den får man `UnicodeEncodeError`.

In [ ]:
with open("dishes.md", "w", encoding="utf-8") as file:
    file.write(result.output)

### Test: be om ett annat format i user-prompten

Här ber användaren om svaret "som ett Python-script", vilket krockar med formatet i `instructions`. Testet visar hur modellen prioriterar när system-instruktionen och user-prompten säger olika saker.

In [ ]:
result = await chef_agent.run("Give me 3 dishes in a list in as a python script.")
print(result.output)

## Storyteller – styra modellen med ModelSettings

`ModelSettings` styr *hur* modellen genererar text, oberoende av vad prompten säger:
- `temperature=0`: modellen väljer nästan alltid det mest sannolika nästa token. Svaren blir förutsägbara och nästan identiska mellan körningar. Högre värden ger mer variation och "kreativitet".
- `thinking="low"`: begränsar hur mycket modellen resonerar internt innan den svarar. Det ger färre tokens, lägre kostnad och snabbare svar.

`timeout=60` gör att anropet avbryts om det tar längre än 60 sekunder.

`instructions` sätter rollen (barnboksberättare) och en längdgräns (max 3 meningar).

In [ ]:
from pydantic_ai import ModelSettings

storyteller = Agent(
    model, instructions="You are a children storyteller, answer max 3 sentences",
    model_settings=ModelSettings(thinking="low", temperature=0)
)

prompt = "Three friends called Lisa, Miranda and Alexandra likes to program Python ..."

story1 = await storyteller.run(prompt, timeout=60)
print(story1.output)

### Tokenanvändning

`.usage` visar hur många tokens anropet förbrukade:
- **input tokens**: prompt + instruktioner
- **output tokens**: modellens svar

Tokens styr både **kostnad** och **latency**, och är därför ett av de viktigaste mätvärdena att följa i LLMOps.

In [ ]:
story1.usage

In [ ]:
story1.all_messages()

# Multimodal

## Multimodal – modell som förstår bilder

För att analysera bilder behövs en **multimodal modell** (LMM). `vl` i modellnamnet står för *vision-language*.

Nvidias API är **OpenAI-kompatibelt**. Därför används `OpenAIChatModel` och `OpenAIProvider`, och man byter bara `base_url` till Nvidias endpoint. API-nyckeln hämtas från en miljövariabel, så den aldrig hamnar i koden.

In [26]:
model = OpenAIChatModel(
    model_name ="nvidia/nemotron-3.5-super-vl-120b-a12b",
    provider = OpenAIProvider(
        base_url = "https://integrate.api.nvidia.com/v1", api_key =os.environ["NVIDIA_API_KEY"]
    ),
)

### Skicka en bild till agenten

- `Path("cat.jpg").read_bytes()` läser bilden som rå bytes.
- `BinaryContent` paketerar bytes tillsammans med `media_type`, så att modellen vet vilken filtyp det är.
- `user_prompt` kan vara en **lista** med text och bild. Modellen tar emot båda i samma meddelande.

Under huven görs bilden om till embeddings i samma vektorrum som texten. Därför kan modellen koppla ihop "this cat" med innehållet i bilden.

In [27]:
from pathlib import Path
from pydantic_ai.messages import BinaryContent

image_joker = Agent(
    model=model,
    instructions="You are a person that can analyze an image, explain the contents and make a joke about it, max 3 sentences.",
)

image_data = Path("cat.jpg").read_bytes()
prompt = "Tell me about this cat"

result = await image_joker.run(
    user_prompt=[prompt, BinaryContent(data = image_data, media_type = "image/jpg")]
)

result.output

"This image features a tabby cat dressed up in wire-rimmed glasses and a bow tie, sitting behind a laptop as if it's working. The cat looks very serious and intellectual with its paws resting on the keyboard. I bet it's busy typing up a dissertation on the most efficient ways to wake its owner up at 4 AM."